## SETUP LOGFIRE

In [1]:
import os, time, warnings
warnings.filterwarnings("ignore")


import logfire 
from dotenv import load_dotenv

load_dotenv()


# Verify keys
print("LOGFIRE_TOKEN  :", "✅" if os.getenv("LOGFIRE_TOKEN")  else "❌  missing")
print("GROQ_API_KEY   :", "✅" if os.getenv("GROQ_API_KEY")   else "❌  missing")
print("GEMINI_API_KEY :", "✅" if os.getenv("GEMINI_API_KEY") else "❌  missing")

LOGFIRE_TOKEN  : ✅
GROQ_API_KEY   : ✅
GEMINI_API_KEY : ✅


In [2]:
import logfire

logfire.configure()
logfire.info('hello , {place}!', place = 'world')

Logfire project URL: https://logfire-us.pydantic.dev/sahilsawant7120/logfire-demo

17:48:56.003 hello , world!


## Simple INFO

In [3]:
logfire.info("notebook_started",
              notebook_name = "pydantic_logfire.ipynb",
              timestamp = time.time(),
              part = "part_1",
              student = "Sahil Sawant",
              tool = "Pydantic Logfire"
              )

17:48:56.014 notebook_started


## Implementing TRACE

In [4]:
with logfire.span("data_processing_simulation", dataset="llm_course", rows=1000):
    logfire.info("step_started", step=1, action="loading data")
    time.sleep(0.3)

    logfire.info("step_started", step=2, action="transforming", columns=12)
    time.sleep(0.2)

    logfire.info("step_started", step=3, action="saving results", output="/tmp/out.csv")

17:48:56.027 data_processing_simulation
17:48:56.028   step_started
17:48:56.329   step_started
17:48:56.532   step_started


## Experiment 2 — Structured Logging with Pydantic Models

In [5]:
from pydantic import BaseModel
from typing import Optional

class LLMRequest(BaseModel):
    user_id: str
    session_id: str
    query: str
    model : str
    temperature: float = 0.7
    max_tokens: Optional[int] = None

class LLMResponse(BaseModel):
    answer: str
    input_tokens: int
    output_tokens: int
    latency_ms : float
    model_used: str

In [6]:
#Passing DATA

request = LLMRequest(
    user_id="user_123",
    session_id="session_456",
    query="What is the capital of France?",
    model="gpt-4",
    temperature=0.5,
    max_tokens=100
)

with logfire.span("llm_request",
                user_id=request.user_id,
                session_id=request.session_id,
                model_used=request.model 
                ):

    logfire.info("request_received" , **request.model_dump())

    time.sleep(0.1)

    response = LLMResponse(
        answer="The capital of France is Paris.",
        input_tokens=10,
        output_tokens=7,
        latency_ms=120.5,
        model_used="llm-4"
    )
    logfire.info("response_generated" , **response.model_dump())

print(response)


17:48:56.632 llm_request
17:48:56.633   request_received
17:48:56.734   response_generated
answer='The capital of France is Paris.' input_tokens=10 output_tokens=7 latency_ms=120.5 model_used='llm-4'


## Experiment 3 - instrumen t Groq 

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage


logfire.instrument_openai()

llm_groq = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    model_name="openai/gpt-oss-20b",
    temperature=0.4,
)

#make an groq request

print("Calling Groq LLM.....")
response = llm_groq.invoke([
    HumanMessage(content="Explain me about the benefits of using Groq for LLM inference in exactly 2 sentences.")
])

print(response.content)

Calling Groq LLM.....
17:49:00.276 Chat Completion with 'openai/gpt-oss-20b' [LLM]
Groq's tensor processing architecture delivers ultra‑low latency and high throughput for LLM inference, enabling real‑time applications with minimal response times. Its energy‑efficient design and cost‑effective scaling reduce operational expenses while maintaining competitive performance against GPU and TPU alternatives.


## Experiment 4 Instrument gemini

In [8]:
llm_gemini = ChatOpenAI(
    base_url = "https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key = os.getenv("GEMINI_API_KEY"),
    model = "gemini-2.5-flash-lite",
    temperature = 0.4,
)

print("Calling Gemini (gemini-2.5-flash-lite)...")
try:
    response = llm_gemini.invoke([
        HumanMessage(content="Explain what an observability 'span' is, in exactly 2 sentences.")
    ])
    print(f"\n🔵 Gemini Response:\n{response.content}")
    print("\n✅ In Logfire dashboard:")
    print("  → You now see BOTH 'openai/gpt-oss-20b' and 'gemini-2.5-flash-lite' in traces")
    print("  → Same query, different providers - compare latency and token usage")
except Exception as e:
    print(f"⚠️ Gemini call failed: {e}")
    print("    Check your GEMINI_API_KEY in .env")

Calling Gemini (gemini-2.5-flash-lite)...
17:56:48.510 Chat Completion with 'gemini-2.5-flash-lite' [LLM]

🔵 Gemini Response:
An observability span represents a single, timed operation within a distributed system, capturing its start time, duration, and any associated metadata. It's a fundamental building block for tracing, allowing you to visualize the flow of requests and diagnose performance bottlenecks across multiple services.

✅ In Logfire dashboard:
  → You now see BOTH 'openai/gpt-oss-20b' and 'gemini-2.5-flash-lite' in traces
  → Same query, different providers - compare latency and token usage


## Experiment 5 - Gemini vs Groq side by side waterfall trace